Linen usage prediction for the next 30 days using Prophet model
> Using Azure ML SDK v2 (MLClient) with Prophet forecasting model

In [ ]:

# Install SDK v2 and Prophet (run once in the notebook if needed)
#!pip install --quiet azure-ai-ml azure-identity prophet mlflow

In [ ]:

%pip show azure-ai-ml

In [2]:
# Connect using MLClient (SDK v2)
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import AmlCompute

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

# MLClient will read configuration from ./config.json or env vars if present
ml_client = MLClient.from_config(credential=credential)
print('MLClient initialized for subscription/workspace')

Found the config file in: /config.json


MLClient initialized for subscription/workspace


In [ ]:

import pandas as pd
df = pd.read_csv('./LinenData.csv')
print(df.columns.tolist())
print(df.head())

['Date', 'Unit', 'DayOfWeek', 'IsHoliday', 'HolidayName', 'HolidayType', 'SurgeriesPerDay', 'PatientsPerDay', 'LinenUsage']
         Date       Unit DayOfWeek  IsHoliday HolidayName HolidayType  \
0  27/11/2022         OR    Sunday      False         NaN         NaN   
1  27/11/2022    MedSurg    Sunday      False         NaN         NaN   
2  27/11/2022        ICU    Sunday      False         NaN         NaN   
3  27/11/2022  Maternity    Sunday      False         NaN         NaN   
4  28/11/2022         OR    Monday      False         NaN         NaN   

   SurgeriesPerDay  PatientsPerDay  LinenUsage  
0                3               3          47  
1                0              56         179  
2                0              11          55  
3                0               9          27  
4               14              14         255  


In [ ]:

target_column_name = 'BlanketUsage'
time_column_name = 'Date'
forecast_horizon = 30
df[time_column_name] = pd.to_datetime(df[time_column_name], format='%d/%m/%Y')
df = df.sort_values(time_column_name).reset_index(drop=True)
print(df.dtypes)
print('Rows:', len(df))

Date               datetime64[ns]
Unit                       object
DayOfWeek                  object
IsHoliday                    bool
HolidayName                object
HolidayType                object
SurgeriesPerDay             int64
PatientsPerDay              int64
LinenUsage                  int64
dtype: object
Rows: 4384


/tmp/ipykernel_120905/2110649403.py:4: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  df[time_column_name] = pd.to_datetime(df[time_column_name])


In [5]:
# Create or get a compute target via MLClient (AmlCompute entity)
compute_name = 'aml-cluster-dev'
existing = {c.name: c for c in ml_client.compute.list()}
if compute_name in existing:
    print('Found compute:', compute_name)
else:
    print('Creating compute:', compute_name)
    compute = AmlCompute(name=compute_name, size='STANDARD_DS11_V2', min_instances=0, max_instances=4)
    ml_client.compute.begin_create_or_update(compute).result()
    print('Compute created')

Found compute: aml-cluster-dev


In [6]:
# Configure MLflow local tracking (change URI to server if you have one)
import mlflow
mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('linen-forecast-experiment')
print('MLflow tracking URI:', mlflow.get_tracking_uri())

MLflow tracking URI: file:./mlruns


In [ ]:
import os
print("Files in current directory:", os.listdir('.'))
if os.path.exists('./data'):
    print("Files in data directory:", os.listdir('./data'))
else:
    print("data/ directory does NOT exist - creating it now")
    os.makedirs('./data', exist_ok=True)
    
    # Copy LinenData.csv to data folder
    import shutil
    shutil.copy('./LinenData.csv', './data/LinenData.csv')
    
    # Create MLTable file with correct format
    mltable_content = """type: mltable
paths:
  - file: ./LinenData.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
"""
    with open('./data/MLTable', 'w') as f:
        f.write(mltable_content)
    
    print("Created data/ folder with MLTable and LinenData.csv")

# Always recreate MLTable with correct encoding if it exists
if os.path.exists('./data/MLTable'):
    mltable_content = """type: mltable
paths:
  - file: ./LinenData.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
"""
    with open('./data/MLTable', 'w') as f:
        f.write(mltable_content)
    print("Updated MLTable file with correct encoding")

Files in current directory: ['.amlignore', '.amlignore.amltmp', '.git', '.gitignore', '.ipynb_aml_checkpoints', 'clean_rg-github.sh', 'consumption_alert.sh', 'data', 'mlruns', 'Run diabetes training script.ipynb', 'Run_linen_training_script.ipynb', 'run_linen_training_script.ipynb.amltmp', 'setup.sh', 'src', 'test.ipynb', 'train-model-deploy.ipynb', 'Training linen.ipynb', 'training linen.ipynb.amltmp', 'usage.csv', 'verify.sh']
Files in data directory: ['MLTable', 'usage.csv']
Updated MLTable file with correct encoding


# Train Prophet Model for Blanket Usage

Now we'll train a Prophet model for the blanket usage time series and log everything to Azure ML using MLflow.

In [ ]:
# Install Prophet if not already installed
import subprocess
import sys

try:
    import prophet
    print("Prophet is already installed")
except ImportError:
    print("Installing Prophet...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "prophet"])
    print("Prophet installed successfully")

# Verify Prophet is available
from prophet import Prophet
print("Prophet version:", prophet.__version__)

Data asset linen_usage_mltable:1764961863 registered successfully
Training data will use: azureml:linen_usage_mltable:1764961863


In [ ]:
# Train Prophet model for blanket usage and log to Azure ML via MLflow
from prophet import Prophet
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import json
import os

# Set MLflow tracking to Azure ML workspace
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)
mlflow.set_experiment('linen-forecast-prophet')

# Start MLflow run for the training process
with mlflow.start_run(run_name="prophet_blanket_forecast") as run:
    mlflow.log_param('model_type', 'Prophet')
    mlflow.log_param('forecast_horizon', forecast_horizon)
    mlflow.log_param('time_column', time_column_name)
    mlflow.log_param('target_column', target_column_name)
    mlflow.log_param('linen_type', 'Blanket')
    
    print(f"\n{'='*60}")
    print(f"Training Prophet model for Blanket Usage")
    print(f"{'='*60}")
    
    # Prepare data in Prophet format (ds, y)
    prophet_df = df[[time_column_name, target_column_name]].copy()
    prophet_df.columns = ['ds', 'y']
    prophet_df = prophet_df.sort_values('ds').reset_index(drop=True)
    
    print(f"Training data size: {len(prophet_df)} days")
    print(f"Date range: {prophet_df['ds'].min()} to {prophet_df['ds'].max()}")
    print(f"Target statistics:")
    print(f"  Mean: {prophet_df['y'].mean():.2f}")
    print(f"  Std: {prophet_df['y'].std():.2f}")
    print(f"  Min: {prophet_df['y'].min():.2f}")
    print(f"  Max: {prophet_df['y'].max():.2f}")
    
    # Create and train Prophet model
    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0
    )
    
    # Add additional regressors if available
    if 'AdmCount' in df.columns:
        model.add_regressor('AdmCount')
        prophet_df['AdmCount'] = df['AdmCount'].values
        print("Added AdmCount as regressor")
    
    if 'DosaCount' in df.columns:
        model.add_regressor('DosaCount')
        prophet_df['DosaCount'] = df['DosaCount'].values
        print("Added DosaCount as regressor")
    
    if 'IsWeekend' in df.columns:
        model.add_regressor('IsWeekend')
        prophet_df['IsWeekend'] = df['IsWeekend'].values
        print("Added IsWeekend as regressor")
    
    model.fit(prophet_df)
    
    # Make future dataframe for forecasting
    future = model.make_future_dataframe(periods=forecast_horizon, freq='D')
    
    # Add regressor values for future dates (using mean for simplicity)
    if 'AdmCount' in df.columns:
        future['AdmCount'] = df['AdmCount'].mean()
    if 'DosaCount' in df.columns:
        future['DosaCount'] = df['DosaCount'].mean()
    if 'IsWeekend' in df.columns:
        # Calculate IsWeekend based on day of week
        future['IsWeekend'] = future['ds'].dt.dayofweek.isin([5, 6]).astype(int)
    
    forecast = model.predict(future)
    
    # Calculate metrics on historical data
    historical_pred = forecast[forecast['ds'].isin(prophet_df['ds'])]
    historical_actual = prophet_df.merge(historical_pred[['ds', 'yhat']], on='ds')
    
    rmse = np.sqrt(mean_squared_error(historical_actual['y'], historical_actual['yhat']))
    mae = mean_absolute_error(historical_actual['y'], historical_actual['yhat'])
    
    # Handle MAPE calculation when y can be zero
    non_zero_mask = historical_actual['y'] != 0
    if non_zero_mask.sum() > 0:
        mape = np.mean(np.abs((historical_actual.loc[non_zero_mask, 'y'] - historical_actual.loc[non_zero_mask, 'yhat']) / historical_actual.loc[non_zero_mask, 'y'])) * 100
    else:
        mape = 0
    
    print(f"\nModel Metrics:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  MAPE: {mape:.2f}%")
    
    # Log metrics to MLflow
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('mape', mape)
    
    # Create visualization
    fig = model.plot(forecast)
    plt.title('Prophet Forecast for Blanket Usage')
    plt.xlabel('Date')
    plt.ylabel('Blanket Usage')
    plt.tight_layout()
    plot_path = 'forecast_plot_blanket.png'
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()
    
    # Create components plot
    fig2 = model.plot_components(forecast)
    plt.tight_layout()
    components_path = 'components_plot_blanket.png'
    plt.savefig(components_path)
    mlflow.log_artifact(components_path)
    plt.close()
    
    # Save forecast to CSV
    forecast_csv_path = 'forecast_blanket.csv'
    future_forecast = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(forecast_horizon).copy()
    future_forecast.to_csv(forecast_csv_path, index=False)
    mlflow.log_artifact(forecast_csv_path)
    
    print(f"\nFuture forecast for Blanket Usage (next {forecast_horizon} days):")
    print(future_forecast.to_string())
    
    # Log model
    mlflow.prophet.log_model(model, "prophet_model_blanket")
    
    print(f"\n{'='*60}")
    print(f"Training completed successfully!")
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"View results in Azure ML Studio: {ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri}")
    print(f"{'='*60}")

Submitted job: cool_feijoa_ymbk5jnnvx
RunId: cool_feijoa_ymbk5jnnvx
Web View: https://ml.azure.com/runs/cool_feijoa_ymbk5jnnvx?wsid=/subscriptions/0ded687b-997b-4159-acf3-9c82e7352f8c/resourcegroups/rg-hhn-dev/workspaces/mlw-hhn-dev-20251127

Execution Summary
RunId: cool_feijoa_ymbk5jnnvx
Web View: https://ml.azure.com/runs/cool_feijoa_ymbk5jnnvx?wsid=/subscriptions/0ded687b-997b-4159-acf3-9c82e7352f8c/resourcegroups/rg-hhn-dev/workspaces/mlw-hhn-dev-20251127



MLflow run id: a6feb3c0cb0d4f36b88edec5e629b95c
